# Stage 4 — Baselines (rebuilt, under the honest evaluator)

We re-establish the floor on the new data, judged by `honest_cv` (feature-cluster holdout + corrupted validation). Same three reference points as before — a trivial baseline, logistic regression, gradient boosting — but now every score reflects the **missing-data reality**.

One model note: gradient boosting (`HistGradientBoostingClassifier`) handles NaN **natively**; logistic regression needs imputation, so it's wrapped with a median `SimpleImputer`.

In [ ]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [3]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [1]:
# ============================================================
# Shared feature engineering — identical for train & test.
# Handles the -9999 no-data values via NaN-aware statistics.
# ============================================================
MONTHS = [f"{i:02d}" for i in range(1, 13)]
BANDS  = ["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]

def load_raw(path):
    """Load a CSV and convert the -9999 no-data sentinel to NaN."""
    return pd.read_csv(path).replace(NODATA, np.nan)

def safe_ratio(a, b):
    # NaN propagates naturally; +0 denominator -> inf, cleaned later
    return (a - b) / (a + b)

def build_features(df):
    """Raw monthly bands -> engineered features.
    NaN-aware: every summary is computed over the VALID months only.
    Adds *_nvalid = how many months actually had data (a quality signal)."""
    feats = {}; idx = {}
    for m in MONTHS:
        g = df[f"green_{m}"]; n = df[f"nir_{m}"]; r = df[f"red_{m}"]; s1 = df[f"swir1_{m}"]
        idx.setdefault("NDWI",  []).append(safe_ratio(g,  n))
        idx.setdefault("MNDWI", []).append(safe_ratio(g,  s1))
        idx.setdefault("NDVI",  []).append(safe_ratio(n,  r))
        idx.setdefault("NDMI",  []).append(safe_ratio(n,  s1))
        idx.setdefault("VHmVV", []).append(df[f"VH_{m}"] - df[f"VV_{m}"])
    def summarize(monthly, name):
        a = np.column_stack([np.asarray(x, float) for x in monthly])  # rows x 12
        feats[f"{name}_mean"]  = np.nanmean(a, 1)
        feats[f"{name}_std"]   = np.nanstd(a, 1)
        feats[f"{name}_min"]   = np.nanmin(a, 1)
        feats[f"{name}_max"]   = np.nanmax(a, 1)
        feats[f"{name}_range"] = np.nanmax(a, 1) - np.nanmin(a, 1)
        feats[f"{name}_med"]   = np.nanmedian(a, 1)
        feats[f"{name}_nvalid"]= np.isfinite(a).sum(1)     # data-quality feature
    for b in BANDS: summarize([df[f"{b}_{m}"] for m in MONTHS], b)
    for k, v in idx.items(): summarize(v, k)
    return pd.DataFrame(feats, index=df.index).replace([np.inf, -np.inf], np.nan)

def corrupt(df, rate, seed=0):
    """Randomly blank band cells to mimic the test set's missingness."""
    rng = np.random.RandomState(seed); d = df.copy()
    bc = [c for c in d.columns if c not in ("ID", "label")]
    arr = d[bc].values.astype(float)
    arr[rng.rand(*arr.shape) < rate] = np.nan
    d[bc] = arr
    return d

In [4]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, roc_auc_score

train = load_raw(TRAIN_PATH)
y = train["label"].values

def honest_cv(make_estimator, train_df, fit_fn=None, n_clusters=10, n_splits=5,
              corrupt_rate=0.6, verbose=True, name=""):
    """Feature-cluster holdout + corrupted validation. The number we trust."""
    yy = train_df["label"].values
    Xfull = build_features(train_df)
    clusters = KMeans(n_clusters=n_clusters, random_state=42,
                      n_init=10).fit_predict(Xfull.fillna(Xfull.median()))
    gkf = GroupKFold(n_splits); oof = np.zeros(len(yy))
    for tr, va in gkf.split(Xfull, yy, clusters):
        Xv = build_features(corrupt(train_df.iloc[va], corrupt_rate, seed=99))
        if fit_fn is None:
            m = make_estimator().fit(Xfull.iloc[tr], yy[tr]); oof[va] = m.predict_proba(Xv)[:,1]
        else:
            oof[va] = fit_fn(train_df.iloc[tr], yy[tr], Xv)
    f1=f1_score(yy,(oof>=.5).astype(int)); auc=roc_auc_score(yy,oof); bl=0.6*f1+0.4*auc
    if verbose: print(f"{name:30s} F1={f1:.4f} AUC={auc:.4f} blend={bl:.4f}")
    return f1, auc, bl

## Baseline 1 — trivial floor

In [5]:
from sklearn.dummy import DummyClassifier
Xc = build_features(train)
def dummy_fit(tr_df, ytr, Xv):
    from sklearn.dummy import DummyClassifier
    m = DummyClassifier(strategy="stratified", random_state=42)
    m.fit(build_features(tr_df), ytr); return m.predict_proba(Xv)[:,1]
_ = honest_cv(None, train, fit_fn=dummy_fit, name="Dummy (random)")

Dummy (random)                 F1=0.3177 AUC=0.4271 blend=0.3615


## Baseline 2 — logistic regression (imputed + scaled)

In [6]:
mk_lr = lambda: make_pipeline(SimpleImputer(strategy="median"),
                              StandardScaler(),
                              LogisticRegression(max_iter=2000))
_ = honest_cv(mk_lr, train, name="Logistic Reg")

Logistic Reg                   F1=0.7084 AUC=0.8426 blend=0.7621


## Baseline 3 — gradient boosting (native NaN)

In [7]:
mk_hgb = lambda: HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
            max_leaf_nodes=7, l2_regularization=1.0, random_state=42)
_ = honest_cv(mk_hgb, train, name="Gradient Boost")

Gradient Boost                 F1=0.7703 AUC=0.9237 blend=0.8317


## Stage 4 takeaways
- Floor set under the **honest** (corrupted) evaluator — scores are realistic, not the clean fantasy.
- Gradient boosting leads; logistic is close behind and stays our interpretable/efficient anchor.
- Crucially, these baselines train on **clean** data but are validated on **corrupted** data — so they *understate* what's achievable. Stage 5's headline move (training on simulated missingness) attacks exactly that.

**Next (Stage 5):** the iteration that matters most this time — make the model robust to missing months.